# LightGBM Anomaly & Fraud Detection Model (Optimized)

This notebook trains and optimizes a LightGBM Classifier to detect transactions fraud by loading features directly from MongoDB, balancing the dataset, and performing stratified train-test splits.

In [1]:
import pandas as pd
import numpy as np
import pickle
import os
from sklearn.model_selection import train_test_split, GridSearchCV
from lightgbm import LGBMClassifier
from sklearn.metrics import classification_report, accuracy_score, confusion_matrix
from pymongo import MongoClient

### Step 1: Load dataset from MongoDB (with CSV fallback)

In [2]:
raw_path = r"C:\Users\Sabor\Desktop\project\processed_data\anomaly_features_raw.csv"
anomaly_cols = ['delay_delta', 'Order Item Quantity', 'Sales', 'profit_margin', 'discount_ratio']

print(f"Loading anomaly features from {raw_path}...")
df = pd.read_csv(raw_path)
X = df[anomaly_cols]
y = df['is_fraud']

print(f"Successfully loaded {len(X)} records from the processed dataset.")

Loading anomaly features from C:\Users\Sabor\Desktop\project\processed_data\anomaly_features_raw.csv...
Successfully loaded 180519 records from the processed dataset.


### Step 2: Balance Dataset and Stratified Train-Test Split

In [3]:
# Balance the entire dataset using Random Oversampling of the minority class
df_full = pd.DataFrame(X, columns=anomaly_cols)
df_full['is_fraud'] = y.values

df_majority = df_full[df_full['is_fraud'] == 0]
df_minority = df_full[df_full['is_fraud'] == 1]

df_minority_oversampled = df_minority.sample(len(df_majority), replace=True, random_state=42)
df_balanced = pd.concat([df_majority, df_minority_oversampled])

X_balanced = df_balanced.drop(columns=['is_fraud'])
y_balanced = df_balanced['is_fraud']

# Perform Train-Test Split with stratification on the balanced labels
X_train, X_test, y_train, y_test = train_test_split(X_balanced, y_balanced, test_size=0.2, random_state=42, stratify=y_balanced)

print(f"Original dataset size: {X.shape[0]} samples")
print(f"Balanced dataset size: {X_balanced.shape[0]} samples (50/50 distribution)")
print(f"Training set size: {X_train.shape[0]} samples")
print(f"Test set size: {X_test.shape[0]} samples")

Original dataset size: 180519 samples
Balanced dataset size: 352914 samples (50/50 distribution)
Training set size: 282331 samples
Test set size: 70583 samples


### Step 3: Grid Search Hyperparameter Optimization

In [4]:
param_grid = {
    'n_estimators': [50, 100, 200],
    'max_depth': [3, 5, 7,],
    'learning_rate': [0.01, 0.05, 0.1],
    'num_leaves': [15, 31, 63]
}

print("Running Cross-Validation Grid Search (K-fold = 5) for LightGBM...")
grid_search = GridSearchCV(
    LGBMClassifier(random_state=42, verbose=-1),
    param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1
)
grid_search.fit(X_train, y_train)

print(f"\nBest Hyperparameters: {grid_search.best_params_}")
print(f"Best CV F1-Score: {grid_search.best_score_:.4f}")

Running Cross-Validation Grid Search (K-fold = 5) for LightGBM...

Best Hyperparameters: {'learning_rate': 0.1, 'max_depth': 7, 'n_estimators': 200, 'num_leaves': 63}
Best CV F1-Score: 0.7785


### Step 4: Model Evaluation

In [5]:
best_lgb = grid_search.best_estimator_
y_pred = best_lgb.predict(X_test)
acc = accuracy_score(y_test, y_pred)
cm = confusion_matrix(y_test, y_pred)

print(f"Test Accuracy: {acc * 100:.2f}%")
print("\nConfusion Matrix:")
print(cm)
print("\nDetailed Classification Report:")
print(classification_report(y_test, y_pred))

Test Accuracy: 76.30%

Confusion Matrix:
[[24151 11141]
 [ 5584 29707]]

Detailed Classification Report:
              precision    recall  f1-score   support

           0       0.81      0.68      0.74     35292
           1       0.73      0.84      0.78     35291

    accuracy                           0.76     70583
   macro avg       0.77      0.76      0.76     70583
weighted avg       0.77      0.76      0.76     70583



### Step 5: Save Optimized Model

In [6]:
print("Re-fitting best model on complete dataset for production...")
# Balance the full dataset before final fitting
df_full = pd.DataFrame(X, columns=anomaly_cols)
df_full['is_fraud'] = y.values

df_maj_full = df_full[df_full['is_fraud'] == 0]
df_min_full = df_full[df_full['is_fraud'] == 1]

df_min_full_oversampled = df_min_full.sample(len(df_maj_full), replace=True, random_state=42)
df_full_balanced = pd.concat([df_maj_full, df_min_full_oversampled])

X_balanced_full = df_full_balanced.drop(columns=['is_fraud'])
y_balanced_full = df_full_balanced['is_fraud']

lgb_optimized = LGBMClassifier(**grid_search.best_params_, random_state=42, verbose=-1)
lgb_optimized.fit(X_balanced_full, y_balanced_full)

model_data = {
    "model": lgb_optimized,
    "features": anomaly_cols,
    "cv_score": grid_search.best_score_,
    "test_accuracy": acc,
    "best_params": grid_search.best_params_
}

model_path = r"c:\Users\Sabor\Desktop\project\processed_data\lgb_anomaly_model.pkl"
tmp_path = model_path + ".tmp"
with open(tmp_path, 'wb') as f:
    pickle.dump(model_data, f)
os.replace(tmp_path, model_path)
print(f"\nSuccessfully saved optimized LightGBM model configuration to {model_path}")

Re-fitting best model on complete dataset for production...

Successfully saved optimized LightGBM model configuration to c:\Users\Sabor\Desktop\project\processed_data\lgb_anomaly_model.pkl
